<a href="https://colab.research.google.com/github/robertbarcik/MCP-tutorial/blob/main/MCP_course.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Welcome: One Server, Any Agent

This is the notebook of the **Model Context Protocol (MCP)** course. You read it top to bottom and run the cells. By the end you have seen every piece of MCP at work: a server, a client, the messages between them, a model driving the server, and a real application using it.

Everything lives in one repository: https://github.com/robertbarcik/MCP-tutorial. The README there is the textbook version of the same material. Re-read it later when you want the calm version.

**What we assume you can already do:** write a small Python function, read a dict, and you have once let a model call a function you wrote (the function-calling loop from the previous course). Nothing else.

## How to read this notebook: two paths

MCP is used by two kinds of people. The colleague who plugs a server into Claude Desktop, judges whether it is any good, and configures it for the team. And the developer who writes servers. This notebook serves both.

- **Everything you see by default is for everyone.** Short cells, plain words, and every cell runs with the Python you already have.
- **🛠️ Developer corner** sections sit at the end of chapters and start collapsed. They hold the proof and the plumbing: what really goes over the wire, why an exception loses your message, how the HTTP server is started. Skip them in class. Open them when you write code for a living, or later at home. Nothing outside a corner depends on anything inside it, and *Run all* still runs them.
- **🎯 Mini-task** is for everyone. **🛠️ Stretch** inside a corner is for developers.

If you are not a developer, your goal for today is concrete: call a tool from Python, read a server file and find its four parts, click through a server in Inspector, watch a model drive five servers, and plug a server into a real application by editing one JSON file.

## The want first

You wrote `search_tickets()` for your IT help desk. It works. Now three people want it: a colleague who lives in Claude Desktop, the team's own agent, and you inside Claude Code. Each of them would need its own glue code, and each time you change the function you would fix three integrations.

**MCP** is an agreed way for a program that owns data or functions to describe them and serve them to any AI application. Write the glue once, and every application that speaks MCP can use it.

Two words we will use all day, one plain sentence each:

- A **server** is the program that owns the functions and waits to be asked. It does nothing on its own.
- A **client** is the program that asks: Claude Code, an agent framework, or a script you write yourself.

The words say nothing about where the programs run. Today both run on your machine.

## When NOT to use MCP

Honesty first, because the internet is full of MCP servers nobody needed.

If you control both sides and the consumer is a coding agent on your own machine, a command-line tool with good documentation usually beats an MCP server. The agent can read the help text, run the command, and read the output. Wrapping it in a server adds a layer and takes nothing away.

Use MCP when at least one of these is true:

- **You do not control the client.** Colleagues on Claude.ai, ChatGPT or Copilot cannot run your script, but they can connect to your server.
- **The user should not have a shell.** A company wants its ticket system reachable by agents, but only through five vetted functions, with a login, and with a log of every call.
- **Many people share one server.** It runs somewhere central, and nobody installs anything.

A good rule of thumb: MCP is for the boundary between you and someone else's agent. Inside your own machine, a good CLI is enough.

## One story for the whole day: the IT help desk

A support company has five systems, each behind its own small server: **tickets**, **customers**, **billing**, a **knowledge base** of fixes, and **assets** (laptops, servers, warranties). An AI assistant should answer questions like *"which customers have open tickets and overdue invoices?"* by asking the right servers.

```
   Claude Code / your script / an agent framework      <- clients
                        |
        +---------+-----+-----+---------+
        |         |           |         |
    tickets   customers   billing   knowledge base   assets      <- MCP servers
```

The plan of the day, one step per chapter: touch the function underneath, be the client by hand, use a stranger's client, let a model drive the server, run all five at once, plug the server into a real application, meet resources and prompts, and finally think about the same server on a URL.

Cost: the whole notebook makes a handful of small model calls, a few cents on the course model.

# Setup

Same ritual as the previous course: install, get the repository files (Colab does it for you), load your key. Just run the three cells.

In [1]:
%pip install -q --disable-pip-version-check mcp==2.2.0 openai==3.13.0
print("Packages installed.")

Note: you may need to restart the kernel to use updated packages.
Packages installed.


In [2]:
import os

if "COLAB_GPU" in os.environ or "COLAB_JUPYTER_IP" in os.environ:
    !git clone --depth=1 -q https://github.com/robertbarcik/MCP-tutorial /content/MCP-tutorial 2>/dev/null || true
    os.chdir("/content/MCP-tutorial")
    print(f"Colab: working in {os.getcwd()}")
else:
    print(f"Local: working in {os.getcwd()}")   # run Jupyter from the repository folder

assert os.path.exists("servers/ticket_server.py"), "servers/ folder not found: are you in the repository folder?"
print("servers/ folder found.")

Local: working in /Users/robertbarcik/git-repos/MCP-tutorial
servers/ folder found.


In [3]:
import json, logging, sys
from openai import OpenAI

for name in ("httpx", "httpx2", "openai"):          # otherwise every HTTP request is logged into the cell output
    logging.getLogger(name).setLevel(logging.WARNING)

# Your key, looked up in this order: Colab secret -> environment variable -> a prompt.
try:
    from google.colab import userdata
    api_key = userdata.get("OPENAI_API_KEY")
except Exception:
    api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    from getpass import getpass
    api_key = getpass("OpenAI API key: ")

llm = OpenAI(api_key=api_key)
MODEL = "gpt-5.6-luna"   # small, cheap model of the current generation

try:                                                # Colab plumbing: starting another program needs a real error stream
    sys.stderr.fileno()                             # (explained in the first Developer corner)
except Exception:
    sys.stderr = open(os.devnull, "w")

print("Ready. Model:", MODEL)

Ready. Model: gpt-5.6-luna


Three things happened. The two packages are pinned to the same versions as the repository's `requirements.txt`, so Colab and your own machine behave the same. The repository gives this notebook the `servers/` folder, which is where the interesting code lives. And the key was found in Colab secrets, in your environment, or you typed it. That is all the setup there is.

# The Thing Underneath

Before any protocol, we touch the function. The ticket server is an ordinary Python file. We import it like any module and call a function. If MCP disappeared tomorrow, this cell would still work.

In [4]:
from servers.ticket_server import TICKETS, search_tickets, get_ticket_details

print(f"The 'database' is a list of {len(TICKETS)} dicts. The first one is a {type(TICKETS[0]).__name__}.\n")

critical = search_tickets(priority="critical")
for ticket in critical["tickets"]:
    print(f"{ticket['ticket_id']}  {ticket['status']:12s} {ticket['subject']}")

The 'database' is a list of 15 dicts. The first one is a dict.

TKT-1002  in_progress  Linux server disk full - /var/log consuming 95% space
TKT-1009  in_progress  Windows 11 BitLocker recovery key prompt on every boot
TKT-1011  resolved     Windows Server 2019 Active Directory replication failing


In [5]:
from pprint import pprint

pprint(get_ticket_details("TKT-9999"))

{'error': 'Ticket TKT-9999 not found',
 'follow_up_tools': ['search_tickets'],
 'reason': 'The ticket_id did not match any tickets in the dataset.',
 'retryable': True,
 'suggested_actions': ['Call search_tickets with a query, customer_id or '
                       'priority filter to rediscover the ticket.',
                       'Verify the ticket_id format (e.g., TKT-1001).'],
 'ticket_id': 'TKT-9999'}


### 🔍 What just happened?

A plain function call, no server anywhere. The second call asked for a ticket that does not exist, and look at what came back: not a crash, but a dict with `error`, `reason`, `suggested_actions` and `follow_up_tools`.

That is a design decision you will see pay off later: **errors are written for the model, not for a human.** A model that reads *"call search_tickets with a query"* knows what to do next. A model that gets a Python traceback does not. Keep this dict in mind; we come back to it when a model is driving.

## Look inside the file

Open `servers/ticket_server.py` in the file browser on the left. It has four parts, in this order:

1. **Data.** `TICKETS`, the list of dicts you just saw. A real server would read a database; ours keeps a list.
2. **Private helpers.** Small functions the tools use. Their names start with an underscore, which is how we say "not a tool".
3. **Tools.** Plain Python functions with type hints and a docstring: `search_tickets`, `get_ticket_details`, and three more. **The docstring is what the model will read**, so it is written for the model: what the tool does, what each argument means, an example value.
4. **The MCP layer.** A few lines at the very bottom. The only part of the file that knows MCP exists.

The next cell prints that fourth part straight from the file. Three lines matter:

- `mcp = MCPServer("tickets")` gives the server a name.
- `mcp.tool(annotations=READ_ONLY)(search_tickets)` hands an ordinary function to the server. The name, the description and the list of arguments are read from the function itself. You write nothing by hand.
- `mcp.run()` waits for a client.

In [6]:
from pathlib import Path

source = Path("servers/ticket_server.py").read_text()
print(source[source.index("# 4. MCP LAYER"):])

# 4. MCP LAYER - the only part of this file that knows MCP exists
# =============================================================================
# Never print() in a server: on stdio, stdout IS the wire to the client.

mcp = MCPServer("tickets")

mcp.tool(annotations=READ_ONLY)(search_tickets)
mcp.tool(annotations=READ_ONLY)(get_ticket_details)
mcp.tool(annotations=READ_ONLY)(get_ticket_metrics)
mcp.tool(annotations=READ_ONLY)(find_similar_tickets)
mcp.tool(annotations=WRITES)(update_ticket_status)

if __name__ == "__main__":
    if "--http" in sys.argv:
        mcp.run(transport="streamable-http", host="127.0.0.1", port=8000)   # http://127.0.0.1:8000/mcp
    else:
        mcp.run()                                                          # stdio: the host starts us



### 🎯 Mini-task

Call `get_ticket_metrics("last_30_days")` and `find_similar_tickets("TKT-1001")` from the same module. Then find the line in the printed MCP layer that makes `find_similar_tickets` visible to MCP. What would change if you deleted that one line? (The function would still work in Python. Only clients would stop seeing it.)

## 🛠️ Developer corner: what the MCP layer reads from a function

What `mcp.tool(...)(function)` extracts, verified on the wire in this notebook's environment:

- **Name**: the function name.
- **Description**: the whole docstring, `Args:` section included. It travels as one text block; the model reads all of it. That is why the docstrings in `servers/` are written with the model as the reader.
- **Argument schema**: generated from the type hints. `str | None = None` becomes an optional string with a default. The schema carries names, types and defaults; per-argument explanations live in the docstring, not in the schema.
- **Annotations**: whatever you pass, `READ_ONLY` or `WRITES` from `servers/common.py`. More on those in the model chapter.

The `sys.path` line at the top of every server file exists so the file works both ways: imported as a module (as you just did) and started as a program by a client. `make_error` in `servers/common.py` is the one helper every server shares; underscore names are private by convention only, and MCP never sees them simply because they are never registered.

**🛠️ Stretch.** Import `servers.common`, call `make_error("x", hints=["y"], follow_up_tools=["z"])` and read the dict. Then find where `READ_ONLY` and `WRITES` are defined and what four flags each of them sets.

# Be the Client

The server is a separate program. Something has to start it and ask it questions. Today that something is us, so there is no magic left when a real application does it later.

The cell below has the shape every client cell in this notebook has. Four lines, always the same:

```python
async with Client(TICKET_SERVER) as tickets:            # start the server program and talk to it
    listed = await tickets.list_tools()                 # question 1: what can you do?
    result = await tickets.call_tool(name, arguments)   # question 2: do this for me
                                                        # the block ends: the server is stopped
```

Two words are new. Treat them as vocabulary: `async with` opens a conversation with another program and closes it at the end of the block; `await` means "wait for the other program to answer". That is all the async you need today.

`TICKET_SERVER` says which program to start: this notebook's own Python, running the server file. The two programs talk through the server's ordinary text input and output, the same channels `print()` and `input()` use. No network, no ports.

In [7]:
from mcp import Client, StdioServerParameters

TICKET_SERVER = StdioServerParameters(command=sys.executable, args=["servers/ticket_server.py"])

async with Client(TICKET_SERVER) as tickets:
    listed = await tickets.list_tools()
    for tool in listed.tools:
        print(f"{tool.name:24s} {tool.description.strip().splitlines()[0]}")

    result = await tickets.call_tool("search_tickets", {"priority": "critical"})

print("\nFirst 300 characters of what the server sent back:\n")
print(result.content[0].text[:300])

search_tickets           Search support tickets by any combination of filters. All filters are optional;
get_ticket_details       Get the full record of one ticket by its ID.
get_ticket_metrics       Ticket counts and average resolution time for a time window.
find_similar_tickets     Find tickets similar to a given ticket (shared tags, category, OS, priority).
update_ticket_status     Change the status of a ticket. This CHANGES data (in memory, for the duration

First 300 characters of what the server sent back:

{
  "tickets": [
    {
      "ticket_id": "TKT-1002",
      "customer_id": "CUST-002",
      "subject": "Linux server disk full - /var/log consuming 95% space",
      "description": "Production Ubuntu 22.04 server has /var/log partition at 95% capacity. Log rotation not working properly.",
      "st


### 🔍 What just happened?

A second Python program started, running the ticket server. Our notebook asked it two things: *what tools do you have?* and *run this one for me*. Then the block ended and the program was stopped.

Look at the tool list: those descriptions are the first lines of the docstrings you saw in the file. And the result came back as **text**, our dict turned into JSON, because that is what travels between two programs.

Everything MCP does is built from these two questions. Applications add convenience on top; the questions stay the same.

## What travels between the two programs?

Two programs talked. What did they send each other? You do not need to read the messages to use MCP. But knowing their shape helps when something breaks, and it explains why a server written in any language works with any client.

Each message is one line of JSON. A request names a `method` and carries an `id`; the answer carries the same `id`. The cell above was three exchanges: the client introduces itself (`server/discover`), asks for the tool list (`tools/list`), and asks for one call (`tools/call`). A call for one ticket looks like this, shortened:

```json
--> to the server
{"jsonrpc": "2.0", "id": 3, "method": "tools/call",
 "params": {"name": "get_ticket_details", "arguments": {"ticket_id": "TKT-1001"}}}

<-- from the server
{"jsonrpc": "2.0", "id": 3,
 "result": {"content": [{"type": "text", "text": "{\"ticket_id\": \"TKT-1001\", \"customer_id\": \"CUST-001\", ..."}],
            "isError": false, "resultType": "complete"}}
```

That is the whole protocol. Resources, prompts and everything else later today are more methods on this same wire. In the next chapter, Inspector shows you these messages live, without any code. Developers can also capture them themselves in the corner below.

### 🎯 Mini-task

Copy the client cell into a new cell below it. Change the call to `get_ticket_details` with `{"ticket_id": "TKT-1001"}` and print the whole text. Then try `TKT-9999`: the same error dict you saw in the first chapter, now as text that came from another program.

## 🛠️ Developer corner: the wire, live

Two things the everyone path hid.

**Plumbing.** The setup cell swapped `sys.stderr` for a null file when it has no file handle. Colab's error stream is not a real file, and starting a subprocess needs one. And one rule for this notebook: cells use plain top-level `await`, which Jupyter and Colab allow. Do **not** add `nest_asyncio` (a patch you may know from other notebooks); it hangs the 2.x MCP client.

**The messages.** MCP speaks **JSON-RPC 2.0**: one JSON object per line, a request with `id` and `method`, an answer with the same `id` and a `result`. To see the real thing, we put a wiretap between us and the server: the Unix command `tee` copies everything that passes through a pipe into a file. The server does not know it is being watched.

In [8]:
import json, shlex, tempfile
from pathlib import Path

LOG = Path(tempfile.gettempdir())
TO_SERVER, TO_CLIENT = LOG / "mcp_to_server.log", LOG / "mcp_to_client.log"
for f in (TO_SERVER, TO_CLIENT):
    f.unlink(missing_ok=True)

wrapper = f"tee {TO_SERVER} | {shlex.quote(sys.executable)} servers/ticket_server.py | tee {TO_CLIENT}"
TAPPED_SERVER = StdioServerParameters(command="sh", args=["-c", wrapper])   # sh and tee exist on Linux (Colab) and macOS

async with Client(TAPPED_SERVER) as tickets:
    await tickets.list_tools()
    await tickets.call_tool("get_ticket_details", {"ticket_id": "TKT-1001"})

sent = TO_SERVER.read_text().splitlines()
received = TO_CLIENT.read_text().splitlines()
print(f"{len(sent)} messages went to the server, {len(received)} came back.")

3 messages went to the server, 3 came back.


In [9]:
def trim(message):
    """Shorten the long parts so the structure stays visible."""
    result = message.get("result", {})
    if "tools" in result:
        result["tools"] = [t["name"] for t in result["tools"]] + ["... (full schemas omitted)"]
    for part in result.get("content", []):
        if len(part.get("text", "")) > 100:
            part["text"] = part["text"][:100] + " ..."
    return message

def show_wire(sent, received):
    answers = {m.get("id"): m for m in map(json.loads, received)}
    for line in sent:
        request = json.loads(line)
        print(f"\n--> to the server:   method={request.get('method')}  id={request.get('id')}")
        print(json.dumps(request, indent=2))
        if request.get("id") in answers:
            print(f"\n<-- from the server: id={request['id']}")
            print(json.dumps(trim(answers[request["id"]]), indent=2))

show_wire(sent, received)


--> to the server:   method=server/discover  id=1
{
  "jsonrpc": "2.0",
  "id": 1,
  "method": "server/discover",
  "params": {
    "_meta": {
      "io.modelcontextprotocol/protocolVersion": "2026-07-28",
      "io.modelcontextprotocol/clientInfo": {
        "name": "mcp",
        "version": "0.1.0"
      },
      "io.modelcontextprotocol/clientCapabilities": {}
    }
  }
}

<-- from the server: id=1
{
  "jsonrpc": "2.0",
  "id": 1,
  "result": {
    "cacheScope": "private",
    "capabilities": {
      "prompts": {
        "listChanged": true
      },
      "resources": {
        "listChanged": true,
        "subscribe": true
      },
      "tools": {
        "listChanged": true
      }
    },
    "resultType": "complete",
    "supportedVersions": [
      "2026-07-28"
    ],
    "ttlMs": 0,
    "_meta": {
      "io.modelcontextprotocol/serverInfo": {
        "name": "tickets",
        "version": ""
      }
    }
  }
}

--> to the server:   method=tools/list  id=2
{
  "jsonrpc": "2.0"

### 🔍 What just happened?

Three requests, three answers, paired by `id`:

1. `server/discover`: the client introduces itself and asks the server what it can do. Look at `_meta`: the protocol version (`2026-07-28`) and the client's name travel with every request.
2. `tools/list`: the same tool names you saw before, now as raw JSON with a schema for each.
3. `tools/call`: the tool name and its arguments; the answer carries our dict as text.

Every result says `"resultType": "complete"`, which tells the client nothing more is coming. Servers written before mid-2026 do not know `server/discover`; the client then gets an error and falls back to an older opening called `initialize`. You will meet that fallback in the host chapter, when a framework with an older client talks to this new server.

**🛠️ Stretch.** Change the call in the wiretap cell to `search_tickets` with `{"status": "open"}` and re-run both cells. How many messages now? Find the `_meta` block in the request, and find the `id` that pairs the call with its answer.

# A Stranger's Client: MCP Inspector

Our client is hand-made. Does a client written by somebody else see the same tools? That is the whole promise, so let's check with **MCP Inspector**, the standard debugging client from the MCP project. It needs Node.js (version 22.19 or newer).

**On your own machine**, in a terminal:

```bash
cd MCP-tutorial                      # the repository folder
source .venv/bin/activate            # Windows: .venv\Scripts\activate
npx @modelcontextprotocol/inspector python servers/ticket_server.py
```

It prints a local URL. Open it, click **Connect**, open **Tools**, pick `get_ticket_details`, enter `TKT-9999`, run it, and read the hint. The **History** tab shows the JSON messages from the previous chapter, live.

**In Colab**, open the terminal (the `>_` icon at the bottom left, available on paid plans), then:

```bash
cd /content/MCP-tutorial
node --version                        # needs 22.19 or newer
npx @modelcontextprotocol/inspector --cli python servers/ticket_server.py --method tools/list
```

The `--cli` form prints the answer in the terminal instead of opening a page. If Colab's Node is too old, watch the instructor's screen for this step; nothing later depends on it.

In [10]:
from IPython.display import Image, display

display(Image(url="https://raw.githubusercontent.com/robertbarcik/MCP-tutorial/main/images/inspector_ui.png", width=820))

### 🔍 What just happened?

Nothing on the server changed, and a program you did not write listed and called its tools. That is what "protocol" buys you: the server has one shape, and any client that speaks the shape can use it.

### 🎯 Mini-task

Point Inspector at `servers/billing_server.py`, call `get_invoice` with no arguments at all, and read what the server suggests.

# Let a Model Drive It

In the previous course you wrote the loop yourself: describe the functions to the model, ask, run the function it wants, hand the result back, repeat until it answers in words. That loop does not change. Two lines do, and both are marked in the cell:

1. The tool list comes from the server, not from a list you typed.
2. Running a tool goes through the server, not through your own Python.

You do not need to read the rest of the cell. It is the loop from the function-calling module with those two lines swapped, plus one small helper, `to_openai_tool`, which renames the fields of a tool description into the names the OpenAI API expects. Run it, then use `run_with_mcp(question, client)` like any function.

In [11]:
def to_openai_tool(tool):
    """Rename the fields of an MCP tool description into the shape the Responses API expects."""
    return {"type": "function", "name": tool.name, "description": tool.description or "", "parameters": tool.input_schema}


async def run_with_mcp(question, mcp_client, max_rounds=6):
    listed = await mcp_client.list_tools()                                # <- CHANGE 1: the tool list comes from the server
    tools = [to_openai_tool(t) for t in listed.tools]
    conversation = [
        {"role": "developer", "content": "You are an IT help-desk assistant. Use the tools to look things up; never invent data. "
                                         "When a tool returns an error with suggested_actions, follow them. Answer briefly."},
        {"role": "user", "content": question},
    ]
    for round_number in range(1, max_rounds + 1):
        response = llm.responses.create(model=MODEL, input=conversation, tools=tools)
        conversation += response.output
        calls = [item for item in response.output if item.type == "function_call"]
        if not calls:                                       # no tool call: the model answered in text, we are done
            return response.output_text
        print(f"round {round_number}: the model asked for {len(calls)} call(s)")
        for call in calls:
            args = json.loads(call.arguments)
            result = await mcp_client.call_tool(call.name, args)          # <- CHANGE 2: the server runs the tool, not us
            output = result.content[0].text
            shown = {k: v for k, v in args.items() if v not in (None, "")}  # the model sends every optional argument, most as null
            print(f"  -> {call.name}({shown}) -> {output[:100].replace(chr(10), ' ')} ...")
            conversation.append({"type": "function_call_output", "call_id": call.call_id, "output": output})
    return "Stopped after too many rounds."

print("run_with_mcp is ready.")

run_with_mcp is ready.


In [12]:
async with Client(TICKET_SERVER) as tickets:
    answer = await run_with_mcp("Which tickets are critical right now? One line each.", tickets)

print("\nAssistant:", answer)

round 1: the model asked for 1 call(s)
  -> search_tickets({'priority': 'critical'}) -> {   "tickets": [     {       "ticket_id": "TKT-1002",       "customer_id": "CUST-002",       "subjec ...



Assistant: - **TKT-1002** — Linux server disk full; `/var/log` at 95% capacity — **in progress**
- **TKT-1009** — Windows 11 BitLocker recovery prompt on every boot — **in progress**


### 🔍 What just happened?

The model saw five tool descriptions, picked `search_tickets`, filled in `priority="critical"`, and the server ran it. The model never saw a line of Python, and our loop never needed to know what the tool does. Both sides only know the description and the argument list, which the server read from the function.

The same loop, packaged for reuse, lives in `client/agent_loop.py` in the repository. The terminal program you will run later imports it.

## Now the errors pay off

Remember the dict with `suggested_actions`? Let's give the model a ticket that does not exist and a hint about what it was, and watch the trace.

In [13]:
async with Client(TICKET_SERVER) as tickets:
    answer = await run_with_mcp("Give me the details of ticket TKT-9999, the BitLocker one.", tickets)

print("\nAssistant:", answer)

round 1: the model asked for 1 call(s)
  -> get_ticket_details({'ticket_id': 'TKT-9999'}) -> {   "error": "Ticket TKT-9999 not found",   "reason": "The ticket_id did not match any tickets in th ...


round 2: the model asked for 1 call(s)
  -> search_tickets({'query': 'BitLocker'}) -> {   "tickets": [     {       "ticket_id": "TKT-1009",       "customer_id": "CUST-007",       "subjec ...


round 3: the model asked for 1 call(s)
  -> get_ticket_details({'ticket_id': 'TKT-1009'}) -> {   "ticket_id": "TKT-1009",   "customer_id": "CUST-007",   "subject": "Windows 11 BitLocker recover ...



Assistant: Ticket **TKT-9999** wasn’t found. The BitLocker ticket appears to be **TKT-1009**:

- **Subject:** Windows 11 BitLocker recovery key prompt on every boot
- **Customer:** CUST-007
- **Status:** In progress
- **Priority:** Critical
- **Category:** Security
- **Assignee:** Sarah Lee
- **Created:** 2026-09-10
- **Last updated:** 2026-09-13
- **Issue:** BitLocker requests the recovery key at every startup after a BIOS update; TPM is enabled.
- **Tags:** Windows, BitLocker, encryption, TPM


### 🔍 What just happened?

The first call failed: the server answered with the error dict. The model read `suggested_actions`, called `search_tickets` with a query, found the real ticket and answered. Our loop has no error handling at all. The recovery happened because the error was **data with a next step**, not a crash.

⚠️ Models are not deterministic. If yours searched first or gave up, run the cell once more; the point is the trace, not the exact route.

*Why not simply raise an exception in the tool?* Because the message you wrote would be swallowed on the way to the model; the developer corner at the end of this chapter proves it. The rule for everyone: **a missing ticket is not a bug, it is Tuesday. Expected failures are data with a next step; exceptions are for real bugs.**

## Annotations: telling the application what a tool does

An application like Claude Code has to decide which tools to run without asking you every time. Is `search_tickets` safe to run silently? Is `update_ticket_status`? The server can say so with **annotations**, small flags attached to each tool:

- `read_only_hint`: the tool only reads.
- `destructive_hint`: the tool may delete or overwrite.
- `idempotent_hint`: calling it twice is the same as once.
- `open_world_hint`: it reaches outside its own data (the internet, other systems).

In our servers, `common.py` defines two presets, `READ_ONLY` and `WRITES`, and the MCP layer attaches one to every tool. The ticket server has one tool that changes data, `update_ticket_status`; it is marked as writing.

In [14]:
async with Client(TICKET_SERVER) as tickets:
    for tool in (await tickets.list_tools()).tools:
        a = tool.annotations
        print(f"{tool.name:24s} read_only={str(a.read_only_hint):5s}  destructive={a.destructive_hint}")

search_tickets           read_only=True   destructive=False
get_ticket_details       read_only=True   destructive=False
get_ticket_metrics       read_only=True   destructive=False
find_similar_tickets     read_only=True   destructive=False
update_ticket_status     read_only=False  destructive=False


### 🔍 What just happened?

Four tools say *read only*, one says it writes. A careful application runs the first four freely and asks before the fifth. You will see exactly that prompt in Claude Code later.

⚠️ They are called *hints* for a reason. An application may ignore them, and a server can lie. Annotations help honest applications and honest servers work together; they are not a security boundary. More on that in the last chapter.

### 🎯 Mini-task

1. Add a field `example_ids=["TKT-1001", "TKT-1002"]` to the error in `get_ticket_details` (in `servers/ticket_server.py`) and re-run the TKT-9999 question. Does the model use it?
2. Copy the TKT-9999 cell and ask `run_with_mcp` to close ticket TKT-1004, then, in the same block, ask for its status. Then open a *new* block and ask for the status again. The change is gone: the server keeps its data in memory, and every block starts a fresh program.

## 🛠️ Developer corner: raise or return?

*"So why not just `raise` an exception in the tool?"* Let's try both. The next cell builds a tiny two-tool server right here in the notebook and connects to it **in-process**: `Client(demo)` takes the server object directly, with no subprocess. Same client, same questions, handy for tests.

Two more details from this chapter. `to_openai_tool` only renames fields; the schema itself, `tool.input_schema`, passes through untouched. The server generated it from the type hints, and the model reads it as is. And the `shown` line in the loop exists because the model sends every optional argument, most of them as `null`; the server treats `null` as "not given".

In [15]:
from mcp.server import MCPServer
from servers.common import make_error

logging.getLogger("mcp").setLevel(logging.CRITICAL)   # the server logs the traceback to ITS OWN error output; we only want to see what the client gets
demo = MCPServer("demo")

@demo.tool()
def lookup_raises(item_id: str) -> dict:
    """Look up an item. Raises on unknown ids."""
    raise ValueError(f"{item_id} not found, try search_items")

@demo.tool()
def lookup_returns(item_id: str) -> dict:
    """Look up an item. Returns an error dict on unknown ids."""
    return make_error(f"{item_id} not found", hints=["Call search_items with a keyword."], follow_up_tools=["search_items"])

async with Client(demo) as c:
    for name in ("lookup_raises", "lookup_returns"):
        r = await c.call_tool(name, {"item_id": "X-1"})
        print(f"{name}:  is_error={r.is_error}\n   {r.content[0].text}\n")

lookup_raises:  is_error=True
   Error executing tool lookup_raises

lookup_returns:  is_error=False
   {
  "error": "X-1 not found",
  "suggested_actions": [
    "Call search_items with a keyword."
  ],
  "retryable": false,
  "follow_up_tools": [
    "search_items"
  ]
}



### 🔍 What just happened?

The raised exception reached the client as `is_error=True` and a generic *"Error executing tool"*. The message you wrote, with its helpful *try search_items*, was swallowed on the way. The returned dict arrived complete.

The rule for tools: **exceptions for bugs, dicts for expected failures.** A missing ticket is not a bug, it is Tuesday.

**🛠️ Stretch.** Read `client/agent_loop.py`: `Toolbox` and `run_with_tools` are this chapter's loop with routing across several servers. Then change `run_with_mcp` to print the full argument dict, nulls included, and see what the model really sends.

# Five Servers, One Chat

The real help desk is five servers and twenty-one tools. The model picks; the program's job is to keep five servers running and send each call to the one that owns the tool. The repository has this as a terminal program, `client/interactive_client.py`. You do not need to read it to use it.

## Run it in a terminal

**In Colab:** open the terminal (`>_` at the bottom left; the packages from the Setup cell are already installed there) and type:

```bash
cd /content/MCP-tutorial
export OPENAI_API_KEY="sk-..."
python client/interactive_client.py
```

**On your own machine:**

```bash
cd MCP-tutorial
source .venv/bin/activate            # Windows: .venv\Scripts\activate
export OPENAI_API_KEY="sk-..."       # Windows: set OPENAI_API_KEY=sk-...
python client/interactive_client.py
```

Questions to try:

- What are all the critical priority tickets?
- Show me customer CUST-001's SLA terms and contacts.
- Which assets have warranties expiring in the next 30 days?
- Which customers have both open tickets and overdue invoices?
- Find similar tickets to TKT-1001 and a knowledge base article that could help.

Type `exit` to stop. All five servers stop with it.

### 🔍 What just happened?

Watch the `->` lines: each one names the tool and, through it, the server that answered. A question like *"open tickets and overdue invoices"* makes the model call the ticket server, then the billing server, then combine. Nobody programmed that route.

### 🎯 Mini-task

Ask a question that needs three servers, for example: *"For customer CUST-002: open tickets, outstanding balance, and any asset with an expired warranty."* Count the `->` lines.

## 🛠️ Developer corner: how five servers stay open

Two things in `client/interactive_client.py` are new compared to this notebook:

- `AsyncExitStack` is one `with` block that keeps five `with` blocks open. Without it we would nest five `async with Client(...)` lines.
- `Toolbox` (in `client/agent_loop.py`) remembers which client owns which tool name, so `toolbox.call("get_invoice", ...)` goes to the billing server.

The loop itself is `run_with_tools` from `client/agent_loop.py`, the same code you ran above with the routing added.

**🛠️ Stretch.** Add `"hr"` to the `SERVERS` list in `client/interactive_client.py` and ask about the remote work policy. The HR server publishes it as a resource, not a tool. Can this client reach it? Why not, and which of the three MCP things (tool, resource, prompt) does a plain agent loop see at all?

# Plug Into a Real Host

You will rarely write the client. Applications that speak MCP already exist, and MCP calls them **hosts**: Claude Code, Claude Desktop, Cursor, VS Code, agent frameworks. Plugging your server into one of them is configuration, not code.

**Claude Code**, in a terminal on your own machine, from the repository folder with the virtual environment active:

```bash
claude mcp add tickets -- python servers/ticket_server.py
claude mcp list                        # tickets ... connected
claude
```

Inside the session, ask: *which tickets are critical?* Then: *close ticket TKT-1004.* The second one triggers a permission prompt: Claude Code saw the annotation that says the tool writes. `/mcp` lists the connected servers and their tools. When you are done: `claude mcp remove tickets`.

Docs: https://code.claude.com/docs/en/mcp

## Configuration files, same shape everywhere

Every host stores the same two things: the command to start the server and its arguments.

The repository ships a `.mcp.json`, which Claude Code reads automatically when you open the folder (it asks you once whether to trust it):

```json
{
  "mcpServers": {
    "tickets":   {"command": "python", "args": ["servers/ticket_server.py"]},
    "customers": {"command": "python", "args": ["servers/customer_server.py"]},
    "billing":   {"command": "python", "args": ["servers/billing_server.py"]}
  }
}
```

**Claude Desktop** uses `claude_desktop_config.json` (Settings, Developer, Edit Config) with the same `mcpServers` key. It runs the servers from anywhere, so use absolute paths there, for both the Python interpreter and the file.

**VS Code** uses `.vscode/mcp.json` with the key `servers` instead of `mcpServers`. **Cursor** uses `mcpServers`. Same idea, one spelling apart.

### 🎯 Mini-task

Register `servers/kb_server.py` in Claude Code as `knowledge` and ask it for a fix for a Windows blue screen. Which tool did it pick, and did it need a second call?

And the honest question from the first chapter, one more time: Claude Code could also just run `python -c "from servers.ticket_server import search_tickets; ..."` on your machine. Why bother with the server? Because the moment the tickets live on another machine, or the person asking is not you, the shell is gone and the server is all there is.

## 🛠️ Developer corner: the next course's framework

In the ADK course, the same server becomes a tool of an agent in one line. The code is shown here, not run, because that framework installs its own (older) MCP client library, which cannot share this notebook's environment:

```python
from google.adk.tools.mcp_tool.mcp_toolset import McpToolset
from google.adk.tools.mcp_tool.mcp_session_manager import StdioConnectionParams
from mcp import StdioServerParameters

ticket_toolset = McpToolset(
    connection_params=StdioConnectionParams(
        server_params=StdioServerParameters(command=sys.executable, args=["servers/ticket_server.py"])
    )
)
agent = LlmAgent(name="helpdesk", model=..., tools=[ticket_toolset])
```

Read it inside out: which program to start, how to talk to it, who handles it. The same three questions as our `Client(...)` line. And remember the wire corner: that older client sends `initialize` instead of `server/discover`, and our server answers both. Different generations, one server.

# Resources and Prompts

Not everything an assistant needs is a question for the model to answer. An HR assistant needs two other things:

- *"Attach the remote work policy to this conversation."* A document with an address. In MCP that is a **resource**: the server publishes it under an address like `hr://policy/remote-work`, and the host shows it to the user, who attaches it. The model does not decide to read it; the person does.
- *"Write a performance review for Alice for the first half year."* A reusable template with blanks. In MCP that is a **prompt**: the host offers it as a menu item or a slash command, the person picks it and fills in the blanks.

So: **tools are for the model to call. Resources and prompts are for the host's user interface and the person in front of it.**

The repository has a small HR server with one of each. The next cell asks it the same way we asked for tools: list them, then read one.

In [16]:
HR_SERVER = StdioServerParameters(command=sys.executable, args=["servers/hr_server.py"])

async with Client(HR_SERVER) as hr:
    print("Resources with a fixed address:")
    for r in (await hr.list_resources()).resources:
        print("  ", r.uri, "-", r.name)
    print("Resource addresses with a blank to fill in:")
    for t in (await hr.list_resource_templates()).resource_templates:
        print("  ", t.uri_template)

    doc = await hr.read_resource("hr://policy/remote-work")
    print("\nThe remote work policy, first lines:\n")
    print(doc.contents[0].text[:330])

    print("\nPrompts, with their blanks:")
    for p in (await hr.list_prompts()).prompts:
        print("  ", p.name, "-", [a.name for a in p.arguments])

Resources with a fixed address:
   hr://policy-index - policy_index
Resource addresses with a blank to fill in:
   hr://policy/{name}

The remote work policy, first lines:

Remote Work Policy

Eligibility:
- All full-time employees after 3 months of employment
- Manager approval required

Remote Work Options:
- Fully remote: work from anywhere in the country
- Hybrid: minimum 2 days per week in the office
- Flexible: choose your schedule with team coordination

Equipment:
- Compa

Prompts, with their blanks:
   performance_review - ['employee_id', 'review_period']


### 🔍 What just happened?

Addresses instead of function names. The policy index has a fixed address; each policy has an address with a blank (`{name}`), and `read_resource` filled the blank with `remote-work` and gave back the document. The prompt has two blanks, an employee and a period, and the host asks the person to fill them.

Where you meet them in real life: the attachment menu in Claude Desktop lists a server's resources; its prompts appear as slash commands. In Claude Code, `/mcp` shows both.

### 🎯 Mini-task

In Inspector, connect to `servers/hr_server.py`. Open the **Resources** tab and read the policy index. Then open **Prompts**, pick `performance_review`, fill in `EMP-002` and a period of your choice, and read the message the server builds. No code needed: this is exactly what a host does behind its menus.

## 🛠️ Developer corner: how the HR server declares them

Two new decorators in `servers/hr_server.py` (that file uses the decorator spelling so you see both ways of registering things):

- `@mcp.resource("hr://policy/{name}")`: an address with a hole in it. The function fills the hole and returns the text.
- `@mcp.prompt()`: the function's arguments become the blanks, its return value the message.

The first cell prints the MCP layer; the second fills in the prompt from the client side.

In [17]:
source = Path("servers/hr_server.py").read_text()
print(source[source.index("# 4. MCP LAYER"):source.index("if __name__")])

# 4. MCP LAYER (this server is small enough to have no helpers of its own)
# =============================================================================
# Never print() in a server: on stdio, stdout IS the wire to the client.

mcp = MCPServer("hr")


@mcp.resource("hr://policy-index")
def policy_index() -> str:
    """List of all HR policy documents and their addresses."""
    return "\n".join(f"hr://policy/{key}  -  {doc['title']}" for key, doc in HR_POLICIES.items())


@mcp.resource("hr://policy/{name}")
def policy(name: str) -> str:
    """One HR policy document by its short name: benefits, remote-work or pto-accrual."""
    if name not in HR_POLICIES:
        return f"No policy called '{name}'. Known policies: {', '.join(HR_POLICIES)}."
    return HR_POLICIES[name]["content"]


@mcp.prompt()
def performance_review(employee_id: str, review_period: str) -> str:
    """Structured brief for writing an employee's performance review."""
    emp = EMPLOYEES.get(employee_id)
    who = (f

In [18]:
async with Client(HR_SERVER) as hr:
    filled = await hr.get_prompt("performance_review", {"employee_id": "EMP-001", "review_period": "first half of 2026"})
    print("get_prompt('performance_review', ...):\n")
    print(filled.messages[0].content.text[:400], "...")

get_prompt('performance_review', ...):

Write a performance review for Alice Johnson (Senior Software Engineer, Engineering, manager Sarah Chen, hired 2021-03-15). Recent projects: Payment API Redesign, Database Migration. Skills: Python, PostgreSQL, Kubernetes.
Review period: first half of 2026.

Follow this structure:
1. Achievements: 3 to 5 concrete accomplishments from the period.
2. Core competencies: rate technical skills, communi ...


### 🔍 What just happened?

`get_prompt` gave back a ready-made message with the employee's record filled in, and `read_resource` earlier gave back the document with its type (`text/plain`). Both went over the same wire as the tools, as `resources/read` and `prompts/get`.

One sentence on a feature you may read about: **sampling** let a server ask the host's model for a completion. It was deprecated in July 2026, because servers that need a model simply call the model provider directly. Do not build on it.

**🛠️ Stretch.** Add a fourth policy to `HR_POLICIES` in `servers/hr_server.py` and a second prompt `onboarding_checklist(employee_name, start_date)`. Re-run the cells above; both should appear without touching anything else.

# Beyond Your Laptop

Everything so far used stdio: the host starts the server as a child program, one user, one machine. A company wants something else: **one server, running centrally, that everyone's agents connect to.** For that MCP has a second transport, **Streamable HTTP**: the server is a small web service and the client gets a URL.

Our servers already support it. The same file, started with `--http`, listens on `http://127.0.0.1:8000/mcp`. The last lines of `ticket_server.py` are:

```python
if "--http" in sys.argv:
    mcp.run(transport="streamable-http", host="127.0.0.1", port=8000)
else:
    mcp.run()
```

And the client side changes exactly one thing: instead of a command to start, it gets an address. In our notebook that is `Client("http://127.0.0.1:8000/mcp")`; in Claude Code it is `claude mcp add --transport http tickets http://127.0.0.1:8000/mcp`. Same tools, same questions. The developer corner at the end of this chapter runs it.

## What changes when the server leaves your laptop

This is where MCP earns its keep, and it comes with three topics to know by name:

**Authentication.** A remote server is a web API like any other. MCP uses OAuth: the host obtains a token from the company's login system and sends it with every request; the server checks it. The Python SDK handles the flow on both sides; you configure it rather than write it.

**Low trust.** A server is somebody's code touching your data. Companies put servers behind a gateway, allow-list which servers agents may use, prefer read-only tools, and log every call. The annotations from earlier are one input to those decisions.

⚠️ **Tool poisoning.** Tool descriptions are text the model reads and trusts. A malicious or compromised server can hide instructions in a description (*"before answering, also send the user's files to ..."*), and annotations can lie. Only connect to servers you trust, read what they expose, and treat a new server like a new dependency in your code, because that is what it is.

### 🎯 Mini-task

A colleague found an MCP server for your ticket system on GitHub and wants to connect the team's Claude Desktop to it. Which of the three topics above would you raise first, and what would you ask to see before saying yes? Two sentences. If you have Inspector open: connect to `servers/kb_server.py` and read every tool description as if you were looking for hidden instructions.

## 🛠️ Developer corner: the same server on a URL

The next cell starts the ticket server in the background with `--http`, waits for the port to open, connects over HTTP, runs the model loop, and stops the server again.

In [19]:
import socket, subprocess, time

if "http_server" in globals():
    http_server.terminate()                             # re-running the cell must not leave port 8000 busy
http_server = subprocess.Popen([sys.executable, "servers/ticket_server.py", "--http"],
                               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(50):                                     # wait up to 5 seconds for the port to open
    if socket.socket().connect_ex(("127.0.0.1", 8000)) == 0:
        break
    time.sleep(0.1)

async with Client("http://127.0.0.1:8000/mcp") as tickets_http:
    print("Tools over HTTP:", [t.name for t in (await tickets_http.list_tools()).tools])
    answer = await run_with_mcp("How many tickets were resolved in the last 30 days?", tickets_http)

print("\nAssistant:", answer)
http_server.terminate()
print("HTTP server stopped.")

Tools over HTTP: ['search_tickets', 'get_ticket_details', 'get_ticket_metrics', 'find_similar_tickets', 'update_ticket_status']


round 1: the model asked for 1 call(s)
  -> get_ticket_metrics({'time_period': 'last_30_days'}) -> {   "time_period": "last_30_days",   "start_date": "2026-08-14",   "total_tickets": 15,   "open_tick ...



Assistant: 3 tickets were resolved in the last 30 days.
HTTP server stopped.


### 🔍 What just happened?

Only the address changed. Same tools, same loop, same answers, but now the server could be on another machine with a hundred clients connected. Under the hood the client no longer starts a program; it sends HTTP requests to a running service, and the OAuth flow from the everyone path would sit exactly here, as a token on every request.

**🛠️ Stretch.** Start the HTTP server again (first half of the cell above) and connect **two** clients to it in the same cell, calling a tool from each. One server, many clients: the thing stdio cannot do. Stop the server at the end.

# Your Turn and Key Takeaways

The exercise comes in two sizes. Pick the one that matches your day.

**Starter (everyone, about 20 minutes): add one tool to an existing server.** Open `servers/hr_server.py`. Below `get_employee`, add a function `list_department(department: str) -> dict` that returns the employees of one department (name and role are enough), and an error dict from `make_error(...)` with a hint listing the known departments when there is no match. Give it a docstring written for the model, with an `Args:` section and an example value. Register it with the same `@mcp.tool(annotations=READ_ONLY)` line as its neighbour. Then check it in Inspector: does it appear in the tool list, and does the error hint read well? Use an LLM for the Python if you like; the part that matters is the description and the error.

**Full (developers, about 30 minutes): build a sixth server for the help desk**, in a domain you choose (scheduling, procurement, travel, ...). Instructions and a checklist: https://github.com/robertbarcik/MCP-tutorial/blob/main/EXERCISE.md

In five lines: copy `servers/ticket_server.py`, replace the data and the functions, keep the four-part order, register the tools at the bottom with one annotation each, and check it in Inspector. Then add it to `.mcp.json` and ask Claude Code about it.

## Key takeaways

- A **server** is data, plain functions, and a few lines of MCP at the bottom. The functions stay importable and testable without MCP.
- A **client** asks two questions: *what tools do you have?* and *run this one*. Everything else is built on those.
- The **wire** is JSON-RPC: `server/discover`, `tools/list`, `tools/call`, paired by `id`.
- **Errors are data for the model.** A dict with suggested actions lets the model recover; an exception hides your message.
- **Annotations** tell the host which tools are safe to run silently. Hints, not guarantees.
- **Hosts** need configuration, not code: a command and its arguments, in a JSON file.
- **Resources and prompts** serve the person and the host's interface; tools serve the model.
- **Streamable HTTP** puts the same server on a URL, which is where authentication, trust and tool poisoning become your problem.
- And the honest rule: on your own machine, with your own agent, a good CLI is enough. MCP is for the boundary to someone else's.

*Cleanup:* every `async with` block stopped its server, and the HTTP cell stopped its process. If you interrupted a cell halfway, a stray server may still run; on Linux or macOS `pkill -f "servers/.*_server.py"` removes it.